<a href="https://colab.research.google.com/github/Jeshurun-B/EMA-optimizer-pipeline-v2/blob/main/coolab_notebooks/week5_regression_trees.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 5 — Regression Trees + Defining a Continuous Target
**EMA Crossover ML Project | 10-Week Curriculum (now 11)**
**Date:** July 7, 2026

---

Random Forest was the original Week 5 plan. It's been pushed to Week 6. Diagnosing `class_target_3` last week took most of a week and required tracing a bug back into the labeling pipeline itself — not because you did anything wrong checking it, but because the checks that would normally catch a bad target (feature leakage, class imbalance) don't cover a target that's confounded by its own construction. Before adding another algorithm on top of the existing classification work, this week is spent doing that check *before* modeling instead of after: defining a genuinely new **continuous** target, with validation built in from the start, using the checklist built directly from what last week found.

This week you move from a single Decision Tree Classifier to a **Decision Tree Regressor** — and instead of one continuous target, you're defining and validating **five in parallel**: how much profit, how good the trade is overall, how badly it could go against you, and the optimal timing for both entry and exit. Two of the five (`Optimum_entry` for entry timing, `total_mae` for adverse excursion) are already built from prior weeks; the other three need to be chosen and defended, and all five need to go through the same validation checklist that would have caught last week's `class_target_3` bug before it cost a week.

**How this notebook is built:** Part A is complete, working setup — carried-forward infrastructure, now including `class_target_3_corrected` at its corrected 1% threshold (the retired `class_target_3` is gone; see the session log if you want the full trace). Part B is genuinely yours across all five targets — there is no scaffolded set of assignments, because picking and validating each one *is* this week's actual work. Parts C–F loop across whatever five targets you land on, TODO-scaffolded like every week since Week 4: concept in markdown, structure and TODOs in code, no implementations.

---
# PART A — Setup (working code, heavily commented)

Carried forward from Weeks 1–4, with the target block updated: `class_target_3` is gone, `class_target_3_corrected` and `class_target_bad` are in, both with balance rates printed (the omission that slowed down catching last week's issue).

In [29]:
!pip install -q supabase scikit-learn statsmodels

In [30]:
# ============================================================
# PART A.1 — IMPORTS
# ============================================================
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns, warnings
warnings.filterwarnings('ignore')

from google.colab import userdata
from supabase import create_client

from sklearn.base import clone
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)
print('Environment ready')

Environment ready


In [31]:
pd.set_option('display.max_columns', None)   # don't truncate wide dataframes when printing
pd.set_option('display.max_rows', 100)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
print('Environment ready')

Environment ready


## A.2 — Connect to both databases

In [58]:
main_client = create_client(userdata.get('SUPABASE_URL'), userdata.get('SUPABASE_KEY'))
analytics_client = create_client(userdata.get('ANALYTICS_SUPABASE_URL'), userdata.get('ANALYTICS_SUPABASE_KEY'))
print('Both databases connected')

Both databases connected


## A.3 — Fetch + merge (keep exactly one merged dataframe)

In [59]:
df_main = pd.DataFrame(main_client.table('signals').select('*').eq('status', 'analyzed').execute().data)
for col in ['checked_at_utc', 'time_of_max_price', 'time_of_min_price']:
    if col in df_main.columns:
        df_main[col] = pd.to_datetime(df_main[col], utc=True, errors='coerce')
df_main = df_main.sort_values('checked_at_utc').reset_index(drop=True)

df_analytics = pd.DataFrame(analytics_client.table('crossover_analytics').select('*').execute().data)
for col in ['crossover_utc', 'optimal_entry_utc']:
    if col in df_analytics.columns:
        df_analytics[col] = pd.to_datetime(df_analytics[col], utc=True, errors='coerce')
df_analytics_renamed = df_analytics.rename(columns={'crossover_utc': 'checked_at_utc'})

df = df_main.merge(df_analytics_renamed, on=['checked_at_utc', 'symbol'], how='inner')
df = df.sort_values('checked_at_utc').reset_index(drop=True)
print(f'Merged: {len(df):,} rows | {df.shape[1]} columns')

Merged: 64,915 rows | 60 columns


## A.4 — Signal quality scoring + classification targets (carried forward, updated)

`class_target_3` is **retired** — do not regenerate it (see session log for the full trace: its source column, `mae_percent`, truncates its search window at the MFE peak, mechanically coupling its magnitude to momentum features). `class_target_3_corrected` (built from `total_mae`, the untruncated main-DB column) and `class_target_bad` (compound risk+outcome) replace it, both validated last week.

In [60]:
# # --- Signal quality scoring (Week 1, locked) --- this is the last notebook that this targets will appear in i have created new ones below
# def get_time_decay_score(c):
#     if c <= 20: return 1.0
#     elif c <= 100: return np.exp(-3.5 * (c-20)/80)
#     elif c <= 400: return np.exp(-3.5) * np.exp(-4 * (c-100)/300)
#     else: return 0.0

# def time_cutoff(c):
#     return 1.0 if c <= 20 else (0.5 if c <= 40 else 0)

# def calculate_four_target_scores(row):
#     is_long = str(row['signal_x']).upper() == 'LONG'
#     if is_long:
#         p, r = float(row['max_move_up_pct']), float(row['max_move_down_pct'])
#         ctp, ctmp = int(row['candles_to_max_price']), int(row['candles_to_min_price'])
#         is_btc, is_1d, is_4h = bool(row['btc_trend_bias']), bool(row['htf_1d_bias']), bool(row['htf_4h_bias'])
#     else:
#         p, r = float(row['max_move_down_pct']), float(row['max_move_up_pct'])
#         ctp, ctmp = int(row['candles_to_min_price']), int(row['candles_to_max_price'])
#         is_btc, is_1d, is_4h = not bool(row['btc_trend_bias']), not bool(row['htf_1d_bias']), not bool(row['htf_4h_bias'])

#     r_safe = 0.01 if r <= 0 else r
#     rr = p / r_safe
#     base = round(min(5.0, rr) + 5.0 * get_time_decay_score(ctp), 2)
#     t1 = base

#     if ctmp < ctp and r >= p: t2 = round(base * 0.2, 2)
#     elif r < 0.25: t2 = round(base * 0.1, 2)
#     else: t2 = base

#     pen = base
#     if 'volume_ratio' in row and float(row['volume_ratio']) < 1.0: pen -= 0.8
#     if not is_btc: pen -= 0.5
#     if not is_1d:  pen -= 1.0
#     if not is_4h:  pen -= 1.2
#     if ctmp < ctp and r > 0.75*p: pen -= 0.8
#     t3 = round(max(0, pen), 2)

#     if r < 0.25 or (ctmp < ctp and r >= p): t4 = 0.0
#     else:
#         rr_p = 5.0 if rr>=4 else (rr if rr>=1.5 else ((rr**7)/(1.5**6) if rr>=1.0 else 0.0))
#         t4 = round(5.0*time_cutoff(ctp) + rr_p, 2) if rr_p > 0 else 0.0
#     return t1, t2, t3, t4



# target_cols = ['T1_Pure_Continuous','T2_Soft_Floor','T3_Assumption_Penalty','T4_Control_Punished']
# df[target_cols] = df.apply(lambda r: pd.Series(calculate_four_target_scores(r)), axis=1)
# df['target_special'] = df[target_cols[:3]].min(axis=1)


#for optimal entry
def optimal_entry_candle(row):
    try:
        if pd.isnull(row['optimal_entry_utc']) or pd.isnull(row['checked_at_utc']): return 0.0
        return float((row['optimal_entry_utc'] - row['checked_at_utc']).total_seconds() / 900)
    except: return 0.0
df['Optimum_entry']  = df.apply(optimal_entry_candle, axis=1)

# --- Classification targets ---
mae_abs, mfe_abs = df['mae_percent'].abs(), df['mfe_percent'].abs()
df['target_b']       = (mfe_abs / (mfe_abs + mae_abs + 1e-8)) * (mfe_abs - mae_abs)
df['class_target_1'] = (df['target_b'] > 0.4).astype(int)
df['class_target_2'] = (df['mfe_percent'] > 1).astype(int)

# class_target_3 retired -- NOT regenerated. total_mae uses the untruncated
# main-DB columns (max_move_down_pct / max_move_up_pct), not mae_percent.
# Threshold updated Week 5: 1.0 (was 0.5 at first validation). Re-run the full
# validation ablation at this new threshold -- do not assume the 0.5%-threshold
# numbers (0.874/0.880 AUC, 0.012 volatility gap) carry over unchanged.
is_long = df['signal_x'].astype(str).str.upper() == 'LONG'
df['total_mae'] = np.where(is_long, df['max_move_down_pct'].astype(float), df['max_move_up_pct'].astype(float))
df['class_target_3_corrected'] = (df['total_mae'] > 1.0).astype(int)
df['class_target_bad'] = ((df['total_mae'] > 1.0) & (df['pnl_percent'] < 0.1)).astype(int)

print('Quality columns + targets applied')
for ct in ['class_target_1', 'class_target_2', 'class_target_3_corrected', 'class_target_bad']:
    long_rate  = df[df['signal_x']=='LONG'][ct].mean()*100
    short_rate = df[df['signal_x']=='SHORT'][ct].mean()*100
    print(f'{ct:<28} +%: LONG {long_rate:5.1f}%  SHORT {short_rate:5.1f}%')

Quality columns + targets applied
class_target_1               +%: LONG  37.1%  SHORT  38.9%
class_target_2               +%: LONG  38.1%  SHORT  39.9%
class_target_3_corrected     +%: LONG  29.5%  SHORT  29.3%
class_target_bad             +%: LONG  23.5%  SHORT  23.3%


In [61]:
 # ============================================================
# PART B: Define REG_TARGETS (Final) note that this is suppose to be done later but i am adding it here even if you might still see it down and it's meant for the enginnered features
# ============================================================
import numpy as np
import pandas as pd

# 1. Profit (Raw)
df['target_profit'] = df['mfe_percent']

# 2. Quality (Raw)
df['target_quality'] = df['target_b']

# 3. Danger (Raw)
df['target_danger'] = df['total_mae']

# 4. Entry timing (Raw)
df['target_entry'] = df['Optimum_entry']

# 5. Exit timing (Raw)
is_long = df['signal_x'].astype(str).str.upper() == 'LONG'
df['trade_time'] = np.where(is_long,
                            (df['candles_to_max_price']).astype(float),
                            (df['candles_to_min_price']).astype(float))
df['target_exit'] = df['trade_time']

# Dictionary mapping: name -> (raw_column_name, apply_log1p_in_cv)
REG_TARGETS = {
    'profit':       ('target_profit', True),
    'quality':      ('target_quality', False),
    'danger':       ('target_danger', True),
    'entry_timing': ('target_entry',  True),
    'exit_timing':  ('target_exit',   True),
}

print('Dynamic Targets defined:', {k: v[0] for k, v in REG_TARGETS.items()})

Dynamic Targets defined: {'profit': 'target_profit', 'quality': 'target_quality', 'danger': 'target_danger', 'entry_timing': 'target_entry', 'exit_timing': 'target_exit'}


## A.5 — Feature engineering pipeline (unchanged from Week 3/4)

In [62]:
def safe_ratio(num, den):
    return (num / den.replace(0, np.nan)).replace([np.inf,-np.inf], np.nan).fillna(0.0)

df['FE_adx_x_volume']        = df['adx_ltf'].astype(float) * df['volume_ratio'].astype(float)
df['FE_macd_x_volume']       = df['macd_histogram_ltf'].astype(float) * df['volume_ratio'].astype(float)
df['FE_ema_sep_x_adx']       = df['ema_separation'].astype(float) * df['adx_ltf'].astype(float)
df['FE_rsi_x_htf4h']         = df['rsi_ltf'].astype(float) * df['htf_4h_bias'].astype(float)
df['FE_adx_x_htf1d']         = df['adx_ltf'].astype(float) * df['htf_1d_bias'].astype(float)
df['FE_rsi4h_x_htf1d']       = df['rsi_4h'].astype(float) * df['htf_1d_bias'].astype(float)
df['FE_adx_x_atr_pct']       = df['adx_ltf'].astype(float) * (df['atr_ltf'].astype(float) / df['price'].astype(float))
df['FE_rsi_delta_x_vol']     = df['rsi_ltf'].astype(float).diff(2).fillna(0) * df['volume_ratio'].astype(float)
df['FE_exhaustion_risk']      = (df['rsi_ltf'].astype(float) > 70).astype(int) * df['ema_separation'].astype(float)

df['FE_ema_ratio']            = safe_ratio(df['ema_fast_ltf'].astype(float), df['ema_slow_ltf'].astype(float))
df['FE_price_to_bb']          = safe_ratio(df['atr_pct'].astype(float), df['bb_width_ltf'].astype(float))
df['FE_adx_4h_ratio']         = safe_ratio(df['adx_ltf'].astype(float), df['adx_4h'].astype(float))
df['FE_vol_efficiency_ratio'] = safe_ratio(df['volume_ratio'].astype(float), df['atr_pct'].astype(float))
df['FE_rsi_mtf_ratio']        = safe_ratio(df['rsi_ltf'].astype(float), df['rsi_4h'].astype(float))
df['FE_spread_to_atr_ratio']  = safe_ratio((df['price'].astype(float)-df['ema_fast_ltf'].astype(float)), df['atr_ltf'].astype(float))

df['FE_adx_trending']        = (df['adx_ltf'].astype(float) > 25.0).astype(int)
df['FE_adx_4h_trending']     = (df['adx_4h'].astype(float) > 25.0).astype(int)
df['FE_rsi_overbought']      = (df['rsi_ltf'].astype(float) > 65.0).astype(int)
df['FE_rsi_oversold']        = (df['rsi_ltf'].astype(float) < 35.0).astype(int)
df['FE_rsi_4h_bull']         = (df['rsi_4h'].astype(float) > 55.0).astype(int)
df['FE_high_volume']         = (df['volume_ratio'].astype(float) > 1.5).astype(int)
df['FE_full_htf_align_long'] = ((df['htf_4h_bias'].astype(int)==1) & (df['htf_1d_bias'].astype(int)==1)).astype(int)

# FIXED: was np.where(strict_s.any(), ...) -- a whole-dataset .any() decided a single
# global switch applied to every row, meaning an early row's feature value depended on
# data from anywhere later in the dataset. Deterministic per-row logic instead, no
# dataset-wide lookahead. Drops the "fallback" branch (which existed only to handle the
# case where strict alignment never occurs anywhere in the data) in favor of always
# using the strict, row-local definition.
df['FE_full_htf_align_short'] = ((df['htf_4h_bias'].astype(int)==-1) & (df['htf_1d_bias'].astype(int)==-1)).astype(int)

df['FE_btc_volume_align']    = ((df['btc_trend_bias'].astype(int)==1) & (df['volume_ratio'].astype(float)>1.0)).astype(int)
df['FE_bb_squeeze_regime']   = (df['bb_width_ltf'].astype(float) < df['atr_ltf'].astype(float)).astype(int)
df['FE_momentum_divergence_bear'] = ((df['price'].astype(float) > df['price'].astype(float).shift(3)) &
    (df['rsi_ltf'].astype(float) < df['rsi_ltf'].astype(float).shift(3))).astype(int)

df['FE_session_asia']    = df['hour_of_day'].isin([23,0,1,2,3,4,5,6,7,8]).astype(int)
df['FE_session_london']  = df['hour_of_day'].isin([7,8,9,10,11,12,13,14,15,16]).astype(int)
df['FE_session_ny']      = df['hour_of_day'].isin([13,14,15,16,17,18,19,20,21]).astype(int)
df['FE_session_overlap'] = df['hour_of_day'].isin([13,14,15]).astype(int)

# FIXED: was np.where((day==0).sum() > (day==5).sum()*1.5, ...) -- same whole-dataset
# .sum() lookahead issue as above. Fixed to the standard UTC weekend convention
# (Saturday=5, Sunday=6 under Monday=0 indexing) -- deterministic, no dataset scan.
# CONFIRM this matches your actual day_of_week convention (Monday=0 vs Sunday=0)
# before trusting it; the original .sum() heuristic existed to sidestep exactly this
# ambiguity, just via the wrong mechanism.
df['FE_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

df['FE_overlap_x_volume'] = df['FE_session_overlap'] * df['volume_ratio'].astype(float)
df['FE_asia_x_volume']    = df['FE_session_asia']    * df['volume_ratio'].astype(float)

df['FE_adx_regime'] = pd.cut(df['adx_ltf'].astype(float), bins=[-np.inf,20.0,35.0,np.inf], labels=[0,1,2]).astype(int)
df['FE_rsi_zone']   = pd.cut(df['rsi_ltf'].astype(float), bins=[-np.inf,30.0,45.0,55.0,70.0,np.inf], labels=[0,1,2,3,4]).astype(int)

# REMOVED: FE_optimum_entry_bin -- pd.cut(df['Optimum_entry'], ...). Direct bin of
# Optimum_entry, which IS this week's target_entry_log (log1p(Optimum_entry)).
# This was the leak: 0.8532 single-feature R² on entry_timing was this column
# predicting a binned version of itself. It was also sitting in FEATURES_COMBINED
# for every OTHER target too. Not replaced -- there's no non-circular version of
# "bin the entry-timing target" to keep.

df = df.sort_values(['symbol','checked_at_utc']).reset_index(drop=True)
signal_map = {'LONG':1,'SHORT':-1}
df['FE_prev_signal_dir'] = df.groupby('symbol')['signal_x'].shift(1).map(signal_map).fillna(0).astype(int)

# REPLACED: FE_prev_t3_score (shift(1) of T3_Assumption_Penalty) -- T1-T4 are retired.
# Same causal pattern (previous trade's quality, as context for the current trade),
# rebuilt on target_quality instead. Requires target_quality to already exist in df
# by this point -- move this block to run AFTER Part B's target definitions, or move
# target_quality's definition earlier if you want feature engineering to stay first.
df['FE_prev_quality_score'] = df.groupby('symbol')['target_quality'].shift(1).fillna(0.0)

def streak(s):
    c = s.ne(s.shift(1)).cumsum()
    return s.groupby(c).cumcount() + 1

df['FE_same_dir_streak']    = df.groupby('symbol')['signal_x'].transform(streak).fillna(0).astype(int)
df['FE_signal_gap_candles'] = df['signal_gap_hours'].astype(float) * 4.0
df['FE_prev_bb_squeeze']    = df.groupby('symbol')['FE_bb_squeeze_regime'].shift(1).fillna(0).astype(int)
sig_num = df['signal_x'].map(signal_map).fillna(0).astype(int)
raw_dc = sig_num != df['FE_prev_signal_dir']
raw_dc.loc[df.groupby('symbol').head(1).index] = False
df['FE_dir_changed'] = raw_dc.astype(int)
prev_gap = df.groupby('symbol')['signal_gap_hours'].shift(1).replace(0,np.nan)
df['FE_prev_signal_gap_decay'] = (df['signal_gap_hours'].astype(float)/prev_gap).replace([np.inf,-np.inf],np.nan).fillna(1.0)

df['_tmp_adx'] = df['FE_adx_regime']; df['_tmp_rsi'] = df['FE_rsi_zone']
df = pd.get_dummies(df, columns=['_tmp_adx','_tmp_rsi'], prefix=['DUM_adx','DUM_rsi'], drop_first=True, dtype=int)

FE_FEATURES = [c for c in df.columns if c.startswith('FE_') or c.startswith('DUM_')]
print(f'Engineered features: {len(FE_FEATURES)}')

Engineered features: 48


In [6]:
# Both wrapper-selected lists reference the two removed columns -- strip them out.
# WRAPPER_FIRST_FINAL_LONG  = [f for f in WRAPPER_FIRST_FINAL_LONG  if f not in ('FE_optimum_entry_bin', 'FE_prev_t3_score')]
# WRAPPER_FIRST_FINAL_SHORT = [f for f in WRAPPER_FIRST_FINAL_SHORT if f not in ('FE_optimum_entry_bin', 'FE_prev_t3_score')]

Engineered features: 49


In [48]:
df

,id_x,checked_at_utc,symbol,signal_x,price,signal_gap_hours,prev_signal,ema_fast_ltf,ema_slow_ltf,ema_fast_slope,ema_slow_slope,ema_separation,price_above_both_emas,crossover_candle_strength,adx_ltf,adx_slope,adx_4h,macd_histogram_ltf,macd_histogram_4h,htf_4h_bias,htf_1d_bias,ema_separation_4h,rsi_4h,rsi_ltf,roc_ltf,atr_ltf,atr_pct,bb_width_ltf,price_to_atr,volume_ratio,volume_trend,crossover_volume_ratio,fear_greed_index,btc_trend_bias,hour_of_day,day_of_week,swing_high,swing_low,atr_stop_distance,max_price_after,min_price_after,max_move_up_pct,max_move_down_pct,time_of_max_price,time_of_min_price,candles_to_max_price,candles_to_min_price,status,id_y,signal_y,entry_price,optimal_entry,optimal_entry_utc,mfe_percent,mae_percent,trade_duration,next_crossover_utc,exit_price,pnl_percent,created_at,Optimum_entry,target_b,class_target_1,class_target_2,total_mae,class_target_3_corrected,class_target_bad,target_profit,target_quality,target_danger,target_entry,trade_time,target_exit,FE_adx_x_volume,FE_macd_x_volume,FE_ema_sep_x_adx,FE_rsi_x_htf4h,FE_adx_x_htf1d,FE_rsi4h_x_htf1d,FE_adx_x_atr_pct,FE_rsi_delta_x_vol,FE_exhaustion_risk,FE_ema_ratio,FE_price_to_bb,FE_adx_4h_ratio,FE_vol_efficiency_ratio,FE_rsi_mtf_ratio,FE_spread_to_atr_ratio,FE_adx_trending,FE_adx_4h_trending,FE_rsi_overbought,FE_rsi_oversold,FE_rsi_4h_bull,FE_high_volume,FE_full_htf_align_long,FE_full_htf_align_short,FE_btc_volume_align,FE_bb_squeeze_regime,FE_momentum_divergence_bear,FE_session_asia,FE_session_london,FE_session_ny,FE_session_overlap,FE_weekend,FE_overlap_x_volume,FE_asia_x_volume,FE_adx_regime,FE_rsi_zone,FE_prev_signal_dir,FE_prev_quality_score,FE_same_dir_streak,FE_signal_gap_candles,FE_prev_bb_squeeze,FE_dir_changed,FE_prev_signal_gap_decay,DUM_adx_1,DUM_adx_2,DUM_rsi_1,DUM_rsi_2,DUM_rsi_3,DUM_rsi_4
0,77492,2019-08-23 09:45:00+00:00,BTCUSDT,SHORT,10118.9700,NaN,None,10154.631934,10157.482400,-0.0877,-0.0541,-0.0281,False,0.8712,13.39,0.63,28.04,-6.848631,4.528482,False,False,-0.4325,44.20,42.86,-0.6887,38.670753,0.3822,1.1926,261.67,1.3176,1.5288,1.3176,50,False,9,4,10220.0000,10108.4400,58.006129,10190.0000,10060.0000,0.7019,0.5828,2019-08-23 12:30:00+00:00,2019-08-23 10:00:00+00:00,11,1,analyzed,43726,SHORT,10118.9700,10162.4200,2019-08-23 09:45:00+00:00,0.58,-0.43,11,2019-08-23T12:30:00+00:00,10154.2600,-0.35,2026-06-25T22:02:40.198887+00:00,0.0,0.086139,0,0,0.7019,0,0,0.58,0.086139,0.7019,0.0,1.0,1.0,17.642664,-9.023756,-0.376259,0.00,0.0,0.0,0.051171,0.000000,-0.0,0.999719,0.320476,0.477532,3.447410,0.969683,-0.922194,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0.0,0.0000,0,1,0,0.000000,1,NaN,0,0,1.0,0,0,1,0,0,0
1,77493,2019-08-23 12:30:00+00:00,BTCUSDT,LONG,10184.0000,0.0,SHORT,10141.075402,10139.469446,0.1059,0.0628,0.0158,True,0.0959,20.71,-1.09,26.06,3.769599,25.199977,False,False,-0.1847,54.25,60.68,0.6822,39.675001,0.3896,1.1356,256.69,1.3657,-0.4878,1.3657,50,False,12,4,10190.0000,10071.2600,59.512502,10445.0000,10150.6000,2.5628,0.3280,2019-08-23 15:15:00+00:00,2019-08-23 12:30:00+00:00,11,0,analyzed,43727,LONG,10184.0000,10150.6000,2019-08-23 12:30:00+00:00,2.56,-0.33,33,2019-08-23T20:45:00+00:00,10333.5300,1.47,2026-06-25T22:02:40.38104+00:00,0.0,1.975363,1,1,0.3280,0,0,2.56,1.975363,0.3280,0.0,11.0,11.0,28.283647,5.148141,0.327218,0.00,0.0,0.0,0.080682,19.215399,0.0,1.000158,0.343079,0.794705,3.505390,1.118525,1.081905,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0.0,0.0000,1,3,-1,0.086139,1,0.0,1,1,1.0,1,0,0,0,1,0
2,77494,2019-08-23 20:45:00+00:00,BTCUSDT,SHORT,10333.2300,0.0,LONG,10371.720659,10374.376512,-0.0927,-0.0566,-0.0256,False,0.6651,38.86,-1.83,23.11,-13.514793,44.211029,True,False,0.1136,53.88,44.16,-0.5551,41.677594,0.4033,0.7355,247.93,1.2311,1.1800,1.2311,50,True,20,4,10444.3200,10265.2900,62.516391,10430.0000,10302.8100,0.9365,0.2944,2019-08-23 21:00:00+00:00,2019-08-23 20:45:00+00:00,1,0,analyzed,43728,SHORT,10333.2300,10349.0000,2019-08-23 20:45:00+00:00,0.29,-0.15,3,2019-08-23T21:30:00+00:00,10374.7900,-0.40,2026-06-25T22:02:40.562213+00:00,

## A.6 — Feature sets (unchanged; dead reference already fixed at the source)

In [63]:
FEATURES_ALL = [f for f in [
    'ema_fast_ltf','ema_slow_ltf','ema_fast_slope','ema_slow_slope','ema_separation','price_above_both_emas',
    'crossover_candle_strength','adx_ltf','adx_slope','adx_4h','macd_histogram_ltf','macd_histogram_4h',
    'htf_4h_bias','htf_1d_bias','ema_separation_4h','rsi_4h','rsi_ltf','roc_ltf','atr_ltf','atr_pct',
    'bb_width_ltf','price_to_atr','volume_ratio','volume_trend','crossover_volume_ratio','fear_greed_index',
    'btc_trend_bias','hour_of_day','day_of_week','swing_high','swing_low','atr_stop_distance','signal_gap_hours'
] if f in df.columns]
FEATURES_COMBINED = list(FEATURES_ALL) + [f for f in FE_FEATURES if f not in FEATURES_ALL]

df_long  = df[df['signal_x']=='LONG'].sort_values('checked_at_utc').reset_index(drop=True)
df_short = df[df['signal_x']=='SHORT'].sort_values('checked_at_utc').reset_index(drop=True)

print(f'FEATURES_ALL: {len(FEATURES_ALL)} | FEATURES_COMBINED: {len(FEATURES_COMBINED)}')
print(f'df_long: {len(df_long):,} | df_short: {len(df_short):,}')

FEATURES_ALL: 33 | FEATURES_COMBINED: 81
df_long: 32,445 | df_short: 32,470


## A.7 — Classification CV functions (carried forward, for reference only)

You will not use these two directly this week — they score classification metrics. They stay here because Part B.2's Target Validation Checklist may have you compare a candidate continuous target back against `class_target_1`/`class_target_2`, and because Tuesday's task is to build the regression equivalent yourself, and having the classification pattern to look at (fold logic, cloning, dropna handling) is a reasonable reference, not a template to copy blind — the scoring internals are genuinely different for regression.

In [64]:
def print_cv(results, model_name='Model'):
    print(f'\n=== {model_name} ===')
    for metric, (mean, std) in results.items():
        print(f'  {metric:<12}: {mean:.4f} +/- {std:.4f}')

def cv_with_scaling(model, df_subset, feature_cols, target_col, n_splits=5, gap=0):
    df_c = df_subset[feature_cols+[target_col]].replace([np.inf,-np.inf],np.nan).dropna().copy()
    X, y = df_c[feature_cols].values, df_c[target_col].values
    tscv = TimeSeriesSplit(n_splits=n_splits, gap=gap)
    scores = {k:[] for k in ['accuracy','precision','recall','f1','auc']}
    for tr, te in tscv.split(X):
        Xtr,Xte,ytr,yte = X[tr],X[te],y[tr],y[te]
        if len(np.unique(ytr))<2 or len(np.unique(yte))<2: continue
        sc = StandardScaler()
        Xtr_s, Xte_s = sc.fit_transform(Xtr), sc.transform(Xte)
        m = clone(model); m.fit(Xtr_s, ytr)
        yp, ypr = m.predict(Xte_s), m.predict_proba(Xte_s)[:,1]
        scores['accuracy'].append(accuracy_score(yte,yp))
        scores['precision'].append(precision_score(yte,yp,zero_division=0))
        scores['recall'].append(recall_score(yte,yp,zero_division=0))
        scores['f1'].append(f1_score(yte,yp,zero_division=0))
        scores['auc'].append(roc_auc_score(yte,ypr))
    return {k:(np.mean(v), np.std(v)) for k,v in scores.items()}

print('cv_with_scaling (classification, reference only) defined')

cv_with_scaling (classification, reference only) defined


## A.8 — What this week's regression models will be compared against

You don't have a regression baseline yet — that's Tuesday's deliverable. This cell is just the classification reference points, carried forward, so you're not scrolling back through three notebooks to find them.

In [65]:
baseline_reference = pd.DataFrame([
    {'Week':'Week 3', 'Model':'LogReg',   'Target':'class_target_1',         'Direction':'LONG',  'AUC':0.6849},
    {'Week':'Week 3', 'Model':'LogReg',   'Target':'class_target_2',         'Direction':'LONG',  'AUC':0.7543},
    {'Week':'Week 4', 'Model':'Bagging',  'Target':'class_target_1',         'Direction':'LONG',  'AUC':0.6817},
    {'Week':'Week 4', 'Model':'LogReg',   'Target':'class_target_3_corrected','Direction':'LONG', 'AUC':0.8736},
    {'Week':'Week 4', 'Model':'LogReg',   'Target':'class_target_bad',       'Direction':'LONG',  'AUC':0.7421},
])
print(baseline_reference.to_string(index=False))
print('\nAll classification. Nothing here is directly comparable to this week\'s R²/RMSE work --')
print('that comparability gap is itself worth having in mind for Part B.')

  Week   Model                   Target Direction    AUC
Week 3  LogReg           class_target_1      LONG 0.6849
Week 3  LogReg           class_target_2      LONG 0.7543
Week 4 Bagging           class_target_1      LONG 0.6817
Week 4  LogReg class_target_3_corrected      LONG 0.8736
Week 4  LogReg         class_target_bad      LONG 0.7421

All classification. Nothing here is directly comparable to this week's R²/RMSE work --
that comparability gap is itself worth having in mind for Part B.


---
# PART B — Define and Validate Five Regression Targets (Monday)

### This section is intentionally not scaffolded like the others

Every other part of this notebook gives you structure to fill in. This one doesn't, on purpose — picking and defending each target *is* the work this week exists to make you do properly, the way it wasn't done for `class_target_3`. What follows is the checklist, run five times, not the answers.

### The five questions

| # | Question | Candidate column(s) | Status |
|---|---|---|---|
| 1 | How much profit? | `pnl_percent` (realized) or `mfe_percent` (potential upside) — pick one, document why | Yours to pick |
| 2 | How good is the trade overall? | `target_special` or `target_b` (both already exist, both continuous) | Yours to pick |
| 3 | How badly could it go against me? | `total_mae` (already built, untruncated) | Built — validate at the new 1.0 threshold |
| 4 | What's the optimal entry timing? | `Optimum_entry` (already built, Week 1) | Built — still worth a fresh correlation check against whichever of #1/#2 you pick |
| 5 | What's the optimal exit timing? | `candles_to_max_price` / `candles_to_min_price` (main-DB labeler) | **Construction-consistency check required first** — see below |

### Target 5's extra step, before anything else

`Optimum_entry` (Target 4) comes from the analytics table's own peak-position logic — the `before_peak`/`before_bottom` window from the `optimal_entry` search you audited two weeks ago. `candles_to_max_price`/`candles_to_min_price` come from the **main signals table's separate labeler** — the "next signal" exit-window one, a different script entirely. Before you treat those as your optimal-exit target and pair them with `Optimum_entry` as if they describe the same trade, confirm both labelers agree on what the trade window actually is. If `end_idx` in one doesn't match the "next signal" boundary in the other, Target 4 and Target 5 would be timing two different windows, not two ends of one trade.

### Checklist — run on all five, not just the new ones

Being "already built" (Targets 3 and 4) doesn't exempt a target from re-validation — Target 3's threshold changed since it was last checked, and Target 4 has never been checked against these specific new targets.

1. **Trace the construction.** For 1, 2, and 5 especially — where does the candidate column actually come from, and does its window depend on another outcome's location?
2. **Plot the full distribution** for each of the 5.
3. **Single-feature check** for each — fit `LinearRegression` on your best 3-4 guesses, one at a time, per target.
4. **Ablate against a dominant driver**, if Step 3 flags one for any target (the way volatility explained most of Target 3).
5. **Cross-check all five against each other**, and against `class_target_1`/`class_target_2`. Five targets that all turn out to measure nearly the same thing isn't five targets, the way the pnl-only `class_target_bad` variant wasn't a real fourth target last week.

In [68]:
# TODO: Step 3 -- single-feature LinearRegression checks, all five
# candidate_features = [...]  # your 3-4 best guesses, can differ per target
# for name, col in REG_TARGETS.items():
#     ...fit LinearRegression on [[f]] -> col for each candidate feature, print R² per target
# ==============================================================================
# PART B: STEP 3 — SINGLE-FEATURE LINEAR REGRESSION VALIDATION (REDO)
# ==============================================================================
# ==============================================================================
# PART B: STEP 3 — SINGLE-FEATURE LINEAR REGRESSION VALIDATION (SANITIZED)
# ==============================================================================
from sklearn.linear_model import LinearRegression

# Re-slice df_long to ensure it has all the freshly engineered targets and safe features
df_long = df[df['signal_x'] == 'LONG'].sort_values('checked_at_utc').reset_index(drop=True)

# Re-mapped strictly to available raw and engineered columns (LEAKS REMOVED)
target_feature_guesses = {
    'profit': [
        'adx_ltf',
        'volume_ratio',
        'FE_adx_x_volume',
        'FE_vol_efficiency_ratio'
    ],
    'quality': [
        'crossover_candle_strength',
        'macd_histogram_ltf',
        'FE_macd_x_volume',
        'FE_prev_quality_score'  # Clean replacement for the T3 leak
    ],
    'danger': [
        'atr_pct',
        'bb_width_ltf',
        'atr_stop_distance',
        'FE_exhaustion_risk'
    ],
    'entry_timing': [
        'signal_gap_hours',
        'FE_signal_gap_candles',
        'FE_session_overlap',    # Safe point-in-time replacement for the bin leak
        'hour_of_day'
    ],
    'exit_timing': [
        'atr_pct',
        'FE_vol_efficiency_ratio',
        'FE_bb_squeeze_regime',
        'fear_greed_index'
    ]
}

print("=== SINGLE-FEATURE CV R² BENCHMARK (LONG SIGNALS) ===")

# Loop over each target configuration based on the dynamic REG_TARGETS dictionary
for name, (target_col, is_log) in REG_TARGETS.items():
    print(f"\nTarget: {name.upper()} ({target_col})")
    print("-" * 50)

    candidates = target_feature_guesses[name]

    for feature in candidates:
        # Verify feature exists in df_long
        if feature not in df_long.columns:
            print(f"  Feature: {feature:<28} | Missing from df_long")
            continue

        # Protect raw scoring space from rogue runaway predictions
        bounds = (-50, 50) if name in ['profit', 'danger'] else None

        try:
            metrics = cv_regression(
                model=LinearRegression(),
                df_subset=df_long,
                feature_cols=[feature],  # Single feature list
                target_info=(target_col, is_log),
                scale=True,
                clip_bounds=bounds
            )

            mean_r2, std_r2 = metrics['r2']
            mean_rmse, _ = metrics['rmse']
            mean_mae, std_mae = metrics['mae']
            print(f"  Feature: {feature:<28} | CV R²: {mean_r2:7.4f} +/- {std_r2:.4f} | Raw RMSE: {mean_rmse:.4f} |CV mae : {mean_mae:7.4f} +/- {std_mae:.4f}"  )

        except Exception as e:
            print(f"  Feature: {feature:<28} | Failed execution: {str(e)}")

=== SINGLE-FEATURE CV R² BENCHMARK (LONG SIGNALS) ===

Target: PROFIT (target_profit)
--------------------------------------------------
  Feature: adx_ltf                      | CV R²: -0.0545 +/- 0.0385 | Raw RMSE: 9.8028 |CV mae :  1.7481 +/- 0.2802
  Feature: volume_ratio                 | CV R²: -0.0440 +/- 0.0405 | Raw RMSE: 9.7224 |CV mae :  1.7361 +/- 0.2793
  Feature: FE_adx_x_volume              | CV R²: -0.0432 +/- 0.0412 | Raw RMSE: 9.7150 |CV mae :  1.7350 +/- 0.2802
  Feature: FE_vol_efficiency_ratio      | CV R²: -0.0405 +/- 0.0335 | Raw RMSE: 9.7010 |CV mae :  1.7151 +/- 0.2939

Target: QUALITY (target_quality)
--------------------------------------------------
  Feature: crossover_candle_strength    | CV R²: -0.0327 +/- 0.0239 | Raw RMSE: 7.7460 |CV mae :  1.4206 +/- 0.2591
  Feature: macd_histogram_ltf           | CV R²: -0.0310 +/- 0.0229 | Raw RMSE: 7.7351 |CV mae :  1.4165 +/- 0.2570
  Feature: FE_macd_x_volume             | CV R²: -0.0313 +/- 0.0230 | Raw RMSE: 7.

In [ ]:
# TODO: Step 4 -- ablation against a suspected dominant driver, for any target Step 3 flagged


In [ ]:
# TODO: Step 5 -- cross-check all five REG_TARGETS against each other (correlation matrix)
#       and against class_target_1 / class_target_2 (groupby or point-biserial correlation)

# TODO: write 2-3 sentences per target (as comments) -- did each survive the checklist,
#       and do any two of the five turn out to be measuring nearly the same thing?


---
# PART C — Regression Theory + Linear Regression Benchmark (Tuesday)

### Concept

Classification metrics (accuracy, precision, AUC) don't apply here — there's no "positive class" to a continuous prediction. The equivalents:

- **MSE / RMSE** — average squared error, then square-rooted back to the target's own units for RMSE. Squaring means big misses are punished disproportionately harder than small ones — one prediction off by 5 costs as much as 25 predictions off by 1.
- **MAE** — average absolute error. Every unit of error costs the same regardless of size; more robust to outliers than MSE.
- **R²** — fraction of the target's variance your model explains, relative to just predicting the mean every time. R²=0 means "no better than guessing the average." R² can go negative on a bad enough test fold — that's not a bug, it means the model did *worse* than the mean-guess baseline on that fold.

You'll need a walk-forward CV function for regression before you can benchmark anything — same `TimeSeriesSplit` fold logic as `cv_with_scaling()`/`cv_no_scaling()` above, different metrics inside the loop.

In [67]:
# ============================================================
# TODO: Build cv_regression() -- walk-forward CV for regression models
# ============================================================
# Same skeleton as cv_with_scaling()/cv_no_scaling() in Part A.7:
#   - TimeSeriesSplit(n_splits=..., gap=...)
#   - clone the model each fold, fit on train, predict on test
#   - score with mean_squared_error (and RMSE = sqrt), mean_absolute_error, r2_score
#     instead of accuracy/precision/recall/f1/auc
#   - no need for the len(np.unique(y_tr)) < 2 check -- that was a classification-only
#     guard against a fold with only one class; doesn't apply to a continuous target
#   - decide for yourself: does this need in-fold scaling (like cv_with_scaling) for
#     LinearRegression, or can you skip it (like cv_no_scaling) once you get to the
#     tree in Part D? (Hint: think about what scaling actually changes for each model
#     type -- you answered a version of this question in Week 4.)

# ============================================================
# PART B: Define REG_TARGETS (Dynamic Transform Architecture)
# ============================================================
# ============================================================
# PART C: Walk-Forward CV (Handles On-the-Fly Transformations)
# ============================================================

def cv_regression(model, df_subset, feature_cols, target_info, n_splits=5, gap=0, scale=False, clip_bounds=None):
    """
    Walk-forward cross-validation loop for regression targets.
    target_info: tuple (raw_target_column_name, apply_log1p) from REG_TARGETS
    """
    target_col, is_log_target = target_info

    # Drop rows where target or features are NaN/Inf
    df_c = df_subset[feature_cols + [target_col]].replace([np.inf, -np.inf], np.nan).dropna().copy()
    X, y = df_c[feature_cols].values, df_c[target_col].values

    tscv = TimeSeriesSplit(n_splits=n_splits, gap=gap)
    scores = {'rmse': [], 'mae': [], 'r2': []}

    for tr, te in tscv.split(X):
        Xtr, Xte = X[tr], X[te]
        # ytr and yte are strictly raw values straight from the dataframe
        ytr, yte = y[tr], y[te]

        if scale:
            sc = StandardScaler()
            Xtr = sc.fit_transform(Xtr)
            Xte = sc.transform(Xte)

        m = clone(model)

        if is_log_target:
            # 1. Transform the training target on the fly
            ytr_fit = np.log1p(ytr)

            # 2. Fit the model in log-space and predict
            m.fit(Xtr, ytr_fit)
            preds_log = m.predict(Xte)

            # 3. Compute Duan's Smearing Factor on log-residuals
            train_log_preds = m.predict(Xtr)
            log_residuals = ytr_fit - train_log_preds
            duan_cf = np.mean(np.exp(log_residuals))

            # 4. Inverse transform predictions back to raw units
            preds_raw = (np.exp(preds_log) * duan_cf) - 1

            # Since yte was never transformed, it is already raw!
            yte_raw = yte

            # 5. Apply optional post-prediction clipping
            if clip_bounds is not None:
                preds_raw = np.clip(preds_raw, clip_bounds[0], clip_bounds[1])

        else:
            # Fit, predict, and evaluate strictly in raw space
            m.fit(Xtr, ytr)
            preds_raw = m.predict(Xte)
            yte_raw = yte

        # Score in raw target units
        scores['rmse'].append(mean_squared_error(yte_raw, preds_raw))
        scores['mae'].append(mean_absolute_error(yte_raw, preds_raw))
        scores['r2'].append(r2_score(yte_raw, preds_raw))

    return {k: (np.mean(v), np.std(v)) for k, v in scores.items()}

In [70]:
# ============================================================
# TODO: Linear Regression benchmark -- loop across all five REG_TARGETS
# ============================================================
# For each (name, col) in REG_TARGETS.items(): run cv_regression() with
# LinearRegression() on df_long (FEATURES_COMBINED) and df_short, print RMSE/MAE/R².
# Store results in one dict/dataframe keyed by target name -- this is the benchmark
# table every subsequent model this week has to beat, per target.

# TODO: for name, col in REG_TARGETS.items(): ... run cv_regression, store + print
# ==============================================================================
# PART C: LINEAR REGRESSION BENCHMARK (The Baseline to Beat)
# ==============================================================================
from sklearn.linear_model import LinearRegression
import pandas as pd

# Ensure we are only passing features that survived the sanitization block
feat_set_long = [f for f in FEATURES_COMBINED if f in df_long.columns]
feat_set_short = [f for f in FEATURES_COMBINED if f in df_short.columns]

linreg_benchmarks = {'LONG': {}, 'SHORT': {}}
results_list = [] # Accumulator for building the final visualization table

print("=== MULTI-FEATURE LINEAR REGRESSION BENCHMARKS ===")

# Process both subsets
for direction, df_subset, features in [('LONG', df_long, feat_set_long), ('SHORT', df_short, feat_set_short)]:
    print(f"\n--- {direction} SIGNALS ({len(features)} Features) ---")

    for name, target_info in REG_TARGETS.items():
        # target_info contains the dynamic tuple: (raw_col_name, apply_log1p_in_cv)

        # Apply strict guardrails for profit/danger to prevent exponential runaway predictions
        CLIP_BOUNDS = {
          'profit':       (-50, 50),
          'danger':       (-50, 50),
          'entry_timing': (0, 500),
          'exit_timing':  (0, 500),
          }
        bounds = CLIP_BOUNDS.get(name) if target_info[1] else None  # target_info[1] is is_log
        try:
            metrics = cv_regression(
                model=LinearRegression(),
                df_subset=df_subset,
                feature_cols=features,
                target_info=target_info,
                scale=True,         # CRITICAL: Matrix math fails if multi-feature sets are unscaled
                clip_bounds=bounds
            )

            # Extract the walk-forward means
            rmse_mean, rmse_std = metrics['rmse']
            mae_mean, mae_std = metrics['mae']
            r2_mean, r2_std = metrics['r2']

            # Store in nested dictionary for downstream programmatic access
            linreg_benchmarks[direction][name] = {
                'RMSE': rmse_mean, 'RMSE_std': rmse_std,
                'MAE': mae_mean, 'MAE_std': mae_std,
                'R2': r2_mean, 'R2_std': r2_std
            }

            # Store formatted strings for the visual table
            results_list.append({
                'Direction': direction,
                'Target': name.upper(),
                'CV_R2': f"{r2_mean:7.4f}",
                'CV_MAE': f"{mae_mean:7.4f}",
                'CV_RMSE': f"{rmse_mean:7.4f}"
            })

            print(f"{name.upper():<14} | R²: {r2_mean:7.4f} | MAE: {mae_mean:7.4f} | RMSE: {rmse_mean:7.4f}")

        except Exception as e:
            print(f"{name.upper():<14} | FAILED: {str(e)}")

# Generate the official baseline scorecard
df_linreg_benchmarks = pd.DataFrame(results_list)

print("\n" + "="*55)
print("OFFICIAL BASELINE SCORECARD (MUST BEAT TO VALIDATE ALPHA)")
print("="*55)
print(df_linreg_benchmarks.to_string(index=False))

=== MULTI-FEATURE LINEAR REGRESSION BENCHMARKS ===

--- LONG SIGNALS (81 Features) ---
PROFIT         | R²:  0.1155 | MAE:  1.3861 | RMSE:  8.3489
QUALITY        | R²: -0.0040 | MAE:  1.3806 | RMSE:  7.7599
DANGER         | R²:  0.4304 | MAE:  0.3380 | RMSE:  0.5295
ENTRY_TIMING   | R²:  0.0067 | MAE:  1.5479 | RMSE:  6.2299
EXIT_TIMING    | R²:  0.0156 | MAE:  9.8851 | RMSE: 210.9620

--- SHORT SIGNALS (81 Features) ---
PROFIT         | R²:  0.1359 | MAE:  1.3523 | RMSE:  6.3698
QUALITY        | R²:  0.0632 | MAE:  1.2171 | RMSE:  5.0421
DANGER         | R²:  0.3999 | MAE:  0.3439 | RMSE:  0.5673
ENTRY_TIMING   | R²:  0.0100 | MAE:  1.3766 | RMSE:  5.2517
EXIT_TIMING    | R²:  0.0188 | MAE:  9.2674 | RMSE: 181.3943

OFFICIAL BASELINE SCORECARD (MUST BEAT TO VALIDATE ALPHA)
Direction       Target   CV_R2  CV_MAE  CV_RMSE
     LONG       PROFIT  0.1155  1.3861   8.3489
     LONG      QUALITY -0.0040  1.3806   7.7599
     LONG       DANGER  0.4304  0.3380   0.5295
     LONG ENTRY_TIMING 

---
# PART D — Decision Tree Regressor: Theory + Implementation (Wednesday)

### Concept

A classification tree splits to maximize Gini/entropy reduction; a regression tree splits to maximize **variance reduction** — at each candidate split, it asks "does dividing these rows into two groups make each group's target values more tightly clustered around their own group mean?" A leaf's prediction is just the mean of the training targets that landed in it, not a majority vote.

Same bias-variance shape you saw in Week 4, different y-axis: shallow trees underfit (high bias, both train and CV RMSE are bad and close together), deep trees overfit (train RMSE keeps dropping, CV RMSE stops improving or gets worse, gap widens).

In [75]:
# ============================================================
# TODO: DecisionTreeRegressor vs LinReg benchmark -- all five REG_TARGETS
# ============================================================
# For each (name, col) in REG_TARGETS.items(): fit DecisionTreeRegressor
# (default params + random_state=42) via cv_regression() on df_long / feat_set / col.
# Compare RMSE/MAE/R² directly against that target's LinReg benchmark from Part C.


# TODO: loop, fit + evaluate, store in tree_results
### The "Default Params" Trap (Fact Check)

# ==============================================================================
# PART D: DECISION TREE VS. LINEAR REGRESSION BENCHMARK
# ==============================================================================
from sklearn.tree import DecisionTreeRegressor
import pandas as pd

tree_results = {'LONG': {}, 'SHORT': {}}
comparison_list = []

print("=== DECISION TREE vs. LINREG (WALK-FORWARD CV) ===")

# Process both signal directions
for direction, df_subset, features in [('LONG', df_long, feat_set_long), ('SHORT', df_short, feat_set_short)]:
    print(f"\n{'-'*60}")
    print(f"--- {direction} SIGNALS ({len(features)} Features) ---")
    print(f"{'-'*60}")
    print(f"{'TARGET':<14} | {'MODEL':<10} | {'CV R²':<8} | {'CV MAE':<8} | {'CV RMSE':<8}")
    print("-" * 60)

    for name, target_info in REG_TARGETS.items():
        # Trees train directly on the raw data distribution; override the log flag to False
        raw_target_col = target_info[0]
        tree_target_info = (raw_target_col, False)

        try:
            # max_depth=5 is the structural guardrail against noise memorization
            metrics = cv_regression(
                model=DecisionTreeRegressor(max_depth= None, random_state=42),
                df_subset=df_subset,
                feature_cols=features,
                target_info=tree_target_info,
                scale=False,         # Trees split on pure thresholds; matrix scaling is unnecessary
                clip_bounds=None     # Trees are naturally bounded by their training leaves
            )

            # Extract Tree walk-forward means
            t_rmse, _ = metrics['rmse']
            t_mae, _ = metrics['mae']
            t_r2, _ = metrics['r2']

            # Store in nested dictionary
            tree_results[direction][name] = {
                'RMSE': t_rmse, 'MAE': t_mae, 'R2': t_r2
            }

            # Fetch the baseline LinReg scores from Part C for comparison
            lr_stats = linreg_benchmarks[direction].get(name, {'R2': 0, 'MAE': 0, 'RMSE': 0})
            lr_r2, lr_mae, lr_rmse = lr_stats['R2'], lr_stats['MAE'], lr_stats['RMSE']

            # Print side-by-side comparison
            print(f"{name.upper():<14} | {'LinReg':<10} | {lr_r2:7.4f}  | {lr_mae:7.4f}  | {lr_rmse:7.4f}")
            print(f"{'':<14} | {'Tree (D=5)':<10} | {t_r2:7.4f}  | {t_mae:7.4f}  | {t_rmse:7.4f}")
            print("-" * 60)

            # Append to list for a clean summary dataframe
            comparison_list.append({
                'Direction': direction,
                'Target': name.upper(),
                'LR_R2': round(lr_r2, 4), 'Tree_R2': round(t_r2, 4),
                'LR_MAE': round(lr_mae, 4), 'Tree_MAE': round(t_mae, 4)
            })

        except Exception as e:
            print(f"{name.upper():<14} | FAILED: {str(e)}")

# Generate the final comparison scorecard
df_comparison = pd.DataFrame(comparison_list)

print("\n" + "="*70)
print("ALGORITHM SURVIVAL SCORECARD (LINREG vs. DECISION TREE)")
print("="*70)
print(df_comparison.to_string(index=False))


=== DECISION TREE vs. LINREG (WALK-FORWARD CV) ===

------------------------------------------------------------
--- LONG SIGNALS (81 Features) ---
------------------------------------------------------------
TARGET         | MODEL      | CV R²    | CV MAE   | CV RMSE 
------------------------------------------------------------
PROFIT         | LinReg     |  0.1155  |  1.3861  |  8.3489
               | Tree (D=5) | -14.3304  |  2.0542  | 59.4381
------------------------------------------------------------
QUALITY        | LinReg     | -0.0040  |  1.3806  |  7.7599
               | Tree (D=5) | -1.8661  |  1.7989  | 19.9460
------------------------------------------------------------
DANGER         | LinReg     |  0.4304  |  0.3380  |  0.5295
               | Tree (D=5) | -0.3554  |  0.5007  |  0.9608
------------------------------------------------------------
ENTRY_TIMING   | LinReg     |  0.0067  |  1.5479  |  6.2299
               | Tree (D=5) | -1.5082  |  2.0463  | 15.7818
-----

In [ ]:
# ============================================================
# TODO: Bias-variance curve, depths 1-20, for at least 2 of the 5 targets
# ============================================================
# Pick the 2 targets where LinReg over/underperformed most in Part C. For each:
# for max_depth in range(1, 21): fit DecisionTreeRegressor(max_depth=d), record
# BOTH train RMSE and CV RMSE (via cv_regression). Plot both curves, x=depth, y=RMSE.
# Same read as Week 4: where do the curves diverge, where does CV RMSE flatten?

# TODO: sweep + plot, chosen targets


---
# PART E — Pruning + Regression-Specific Diagnostics (Thursday)

### Concept

Cost-complexity pruning works the same way it did in Week 4 — `cost_complexity_pruning_path()` gives you a sequence of `ccp_alpha` values, each corresponding to a differently-pruned tree, walk-forward CV picks the best one. The diagnostics that follow are new: a regression model's errors have a *sign and a shape*, not just a right/wrong label, and that shape often tells you something a single RMSE number can't.

In [ ]:
# ============================================================
# TODO: Cost-complexity pruning path -- per target
# ============================================================
# For each of the 5: path = DecisionTreeRegressor(random_state=42).cost_complexity_pruning_path(X_train, y_train)
# Sweep ccp_alphas from that path, cv_regression() each one, pick the best by CV RMSE.
# Lock in BEST_TREE_PARAMS per target (a dict of dicts).

BEST_TREE_PARAMS = {}
# TODO: loop over REG_TARGETS, prune, store best params per target


In [ ]:
# ============================================================
# TODO: Residual diagnostics -- pick at least 2 of the 5 targets to go deep on
# ============================================================
# For each chosen target: fit its best tree, predict on a held-out walk-forward
# test fold.
# 1. Scatter: predicted (x) vs actual (y), with a y=x reference line.
# 2. Scatter: residuals (actual - predicted) vs each of your top 3-4 features for
#    that target -- visible structure (slope, curve, fan shape) means the model
#    is still missing something systematic.

# TODO: predicted vs actual plots
# TODO: residuals vs top features plots


In [ ]:
# ============================================================
# TODO: Where does each tree fail worst?
# ============================================================
# For each of the 5: sort by |residual| descending, inspect the top 15-20 rows
# directly. Pattern (symbol/session/regime) or unstructured noise?

# TODO: build residuals dataframe per target, sort, inspect top rows


---
# PART F — Consolidation + Week 5 Honest Record (Friday)

In [ ]:
# ============================================================
# TODO: Final comparison -- LinReg benchmark vs tuned tree, all 5 targets, both directions
# ============================================================
# TODO: build one table: {Target, Model: LinReg/Tuned Tree, Direction: LONG/SHORT,
#       RMSE, MAE, R²} across all five REG_TARGETS and print it


In [ ]:
# ============================================================
# TODO: Week 5 Honest Record
# ============================================================
week5_record = """
=================================================================
        WEEK 5 PRODUCTION LOG — REGRESSION TREES (5 TARGETS)
=================================================================

  TARGETS DEFINED
    1. Profit       -- column: TODO | rejected alternative: TODO
    2. Quality       -- column: TODO | rejected alternative: TODO
    3. Danger        -- total_mae, threshold 1.0 | re-validation vs 0.5 threshold: TODO
    4. Entry timing  -- Optimum_entry | fresh correlation check vs #1/#2: TODO
    5. Exit timing   -- column: TODO | labeler-consistency check with Target 4: TODO

  CROSS-TARGET CHECK
    Correlation matrix across all 5 + class_target_1/2: TODO
    Any two targets measuring nearly the same thing?: TODO

  MODEL RESULTS (RMSE / MAE / R², LONG then SHORT, LinReg -> Tuned Tree)
    Profit:       TODO
    Quality:      TODO
    Danger:       TODO
    Entry timing: TODO
    Exit timing:  TODO

  DIAGNOSTICS
    Easiest target to predict, and why: TODO
    Hardest target to predict, and why: TODO
    Worst-failure pattern(s) found: TODO

  HONEST VERDICT
    Biggest surprise this week: TODO
    Which of the 5 (if any) would you actually use in production, and how: TODO
    Expectation for Week 6 Random Forest: TODO
=================================================================
"""
print(week5_record)